# CutScope: official data audit and exploration

Run cells in order using the **CutScope analytics** kernel. This notebook imports the same modules as the batch builder. Candidate consumption is not recoverable savings. Source data remains in the official repository; do not commit executed outputs. Save executed copies under `generated/` or `analytics/notebooks/local/`.

In [ ]:
from pathlib import Path
import os, sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "PROJECT_SPEC.md").exists() and (p / "analytics").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the CutScope repository")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from analytics.load_data import load_official_data
from analytics.exploration import audit_data, schema_table, findings_table, candidate_diagnostics
from analytics.outcomes import build_summary, weighted_utilization
from analytics.pricing import Pricing
from analytics.opportunities.idle_interactive import idle_candidates
from analytics.build_analysis import build_snapshot

pd.set_option("display.max_columns", 20)
DATA_DIR = os.environ.get("CUTSCOPE_DATA_DIR")  # default: ~/Desktop/hackathon-2026-official/track-2/data
PRICE = Pricing(2.50, "cutscope-assumption-v1")  # scenario assumption, not a verified bill

## 1. Load and verify
Reads prepared tables and findings in place. Checks the official value hashes before using them. An absent checksum manifest is reported as unknown verification.

In [ ]:
data = load_official_data(DATA_DIR)
jobs, gpus = data.jobs, data.gpus
finding_index = findings_table(data.findings)
display(pd.DataFrame(data.provenance["sources"]).T)
print(f"{len(jobs):,} jobs; {len(gpus):,} card rows; {len(data.findings):,} findings")

## 2. Understand the fields and quality
Inspect job and card grains separately. Timestamps are offsets, not calendar dates. `gpu_hours` is measured; `gpu_hours_alloc` is a comparison derived from final walltime. Missing measurements remain unknown. Duration mismatches are diagnostics, not automatically corrected facts.

In [ ]:
display(schema_table(jobs))
display(schema_table(gpus))
audit = audit_data(data)
display(pd.Series(audit, name="value").to_frame())

In [ ]:
comparison = jobs[["id_job", "attempts", "gpu_hours", "gpu_hours_alloc", "gpu_hours_ratio", "walltime_sec"]]
display(comparison.loc[(comparison.gpu_hours_ratio - 1).abs() > 0.1].sort_values("gpu_hours", ascending=False).head(20))
display(comparison.loc[comparison.walltime_sec.isna()])

## 3. Where the money went
Outcome percentages use all measured GPU-hours in this sample. They do not measure fleet utilization or waste. USD uses the stated scenario rate.

In [ ]:
summary = build_summary(jobs, PRICE)
outcomes = pd.DataFrame(summary["outcomes"])
display(outcomes)
print(f"Sample resource value: ${summary["total_cost_usd"]:,.2f} at ${PRICE.rate:.2f}/GPU-hour")
ax = outcomes.set_index("name").gpu_hours.sort_values().plot.barh(figsize=(9, 4), title="Measured GPU-hours by final outcome")
ax.set_xlabel("GPU-hours in sampled jobs")
plt.tight_layout()
plt.show()

## 4. Workload mix and utilization
Compare workload classes and GPU-hour-weighted utilization. Job summaries do not reveal when useful activity happened. This notebook avoids ranking people by waste.

In [ ]:
workloads = jobs.groupby("job_type").agg(job_count=("id_job", "size"), gpu_hours=("gpu_hours", "sum"))
workloads["sample_share_percent"] = 100 * workloads.gpu_hours / jobs.gpu_hours.sum()
display(workloads.sort_values("gpu_hours", ascending=False))
display(pd.Series(weighted_utilization(jobs)))
fig, ax = plt.subplots(figsize=(9, 4))
valid = jobs.sm_util_avg.notna() & jobs.gpu_hours.notna()
ax.hist(jobs.loc[valid, "sm_util_avg"], bins=20, range=(0, 100), weights=jobs.loc[valid, "gpu_hours"])
ax.set(xlabel="Job-average SM utilization (%)", ylabel="Measured GPU-hours", title="Consumption by utilization band")
plt.tight_layout()
plt.show()

## 5. Findings: real, synthetic, historical, and overlapping
Retain historical findings for sample analysis. Active status is a separate operational view. Exclude explicitly synthetic evidence from real-data candidate cohorts. The full hours of an imbalance job include busy cards and cannot be treated as imbalance savings. Do not add these overlapping cohort totals.

In [ ]:
display(finding_index.groupby(["synthetic", "impact_kind", "impact_scope"], dropna=False).size().rename("findings").to_frame())
cohorts, overlaps = candidate_diagnostics(data)
display(cohorts)
display(overlaps)

## 6. Idle-interactive candidates
Recompute the published rule: interactive workload, walltime >4 hours, mean SM <5%. Check agreement with the official findings. These criteria identify investigation candidates; they do not identify an idle start time.

In [ ]:
idle = idle_candidates(jobs)
official_ids = set(finding_index.loc[finding_index.detector_id.eq("rules::idle-interactive-session") & finding_index.synthetic.eq(False), "job_id"].dropna())
recomputed_ids = set(idle.id_job)
assert recomputed_ids == official_ids, "Rule disagreement: investigate before modeling savings"
print(f"{len(idle):,} candidate jobs; {idle.gpu_hours.sum():,.1f} GPU-hours consumed, NOT savings")
display(idle[["id_job", "state_name", "gpu_count", "gpu_hours", "walltime_sec", "sm_util_avg", "sm_util_max", "mem_used_frac", "attempts"]].head(20))
display(idle.groupby("state_name").agg(jobs=("id_job", "size"), gpu_hours=("gpu_hours", "sum")))

## 7. Inspect one job and its evidence
Change `JOB_ID` to investigate another candidate. Preserve upstream finding objects and distinguish measurements from detector interpretations.

In [ ]:
JOB_ID = int(idle.iloc[0].id_job)
display(jobs.loc[jobs.id_job.eq(JOB_ID)].T)
display(gpus.loc[gpus.id_job.eq(JOB_ID)])
job_findings = [f for f in data.findings if (f.get("metadata") or {}).get("job_id") == JOB_ID]
for finding in job_findings:
    display(finding)

## 8. Hardware signal and cancellation context
A successful retry can hide a node failure in final state. Cancellation alone is not waste. This comparison describes durations, not a known delay after a cancellation request.

In [ ]:
failures = jobs.loc[jobs.hit_node_failure]
display(failures[["id_job", "state_name", "attempts", "nodefail_attempts", "nodefail_nodes", "nodefail_exact", "nodefail_wall_sec"]])
cancelled = jobs.loc[jobs.state_name.eq("CANCELLED")].copy()
cancelled["mean_sm_below_5pct"] = cancelled.sm_util_avg.lt(5).where(cancelled.sm_util_avg.notna(), pd.NA)
cancelled["walltime_hours"] = cancelled.walltime_sec / 3600
display(cancelled.groupby("mean_sm_below_5pct", dropna=False).walltime_hours.describe())

## 9. Confirm the batch handoff
The first snapshot contains verified summary values and diagnostic metadata. Opportunities remain empty until savings, primary attribution, confidence and downside assumptions are ready. Generate the JSON using the command in `analytics/README.md`.

In [ ]:
snapshot = build_snapshot(data, PRICE)
assert snapshot["summary"] == summary
assert sum(x["jobs"] for x in summary["outcomes"]) == len(jobs)
assert abs(sum(x["gpu_hours"] for x in summary["outcomes"]) - summary["total_gpu_hours"]) < 1e-6
print("Notebook and batch use identical baseline calculations. No savings scenarios exported yet.")

## Next investigation
Select several zero-peak and nonzero-peak sessions. Decide which are legitimate interactive pauses, verify requeue/duration anomalies, and document pilot policy assumptions before calculating low/point/high reclaim. Keep candidate consumption, modeled reclaim, and cash savings separate.